# Mask Filling (Local Models)

Use this notebook to experiment with Hugging Face `fill-mask` models while storing model files **inside this project workspace**.

In [ ]:
import os
from dotenv import load_dotenv

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_CACHE"] = "./huggingface"
_ = load_dotenv(dotenv_path=".env.local")

In [ ]:
import torch

from transformers import FillMaskPipeline, pipeline
from typing import List

In [ ]:
experiments: List[str] = [
    "The deploy was rolled {mask} after the health checks failed.",
    "Engineers rotated the API keys after the suspected {mask}.",
    "Shares rallied after the company raised its revenue {mask}.",
    "Flights were canceled because of dense morning {mask}.",
    "A squeeze of {mask} made the soup taste more fresh.",
    "The coach switched to a more aggressive {mask} in the second half.",
    "The telescope captured a remarkably clear {mask} of the galaxy.",
    "The {mask} is responsible for managing the team engagements, expectations and work quality."
]

In [ ]:
def get_local_mask_filler(model_id: str) -> FillMaskPipeline:
    device = "cuda" if torch.cuda.is_available() else "cpu"

    print(f"Loading model \"{model_id}\" on {device}")
    classifier = pipeline(task="fill-mask", model=model_id, device=device)

    return classifier

In [ ]:
def run_fill_mask_experiment(model_id: str):
    mask_filler = get_local_mask_filler(model_id)

    print(f"{'=' * 10} [{model_id}] {'=' * 10}")

    mask_token = mask_filler.tokenizer.mask_token
    print(f"Mask token: {mask_token}")

    for i in range(len(experiments)):
        print()

        prompt = experiments[i].format(mask=mask_token)
        print(prompt)
        result = mask_filler(prompt, top_k=3)

        for j in range(len(result)):
            print(f"  - {result[j]["token_str"]:<20} score={result[j]["score"]:.4%}")

In [ ]:
run_fill_mask_experiment("google-bert/bert-base-uncased")

In [ ]:
run_fill_mask_experiment("microsoft/mpnet-base")

In [ ]:
run_fill_mask_experiment("google/electra-base-generator")